# Master Pipeline: LLIE + YOLOv11n on ExDark

**Evaluasi Metode Low-Light Image Enhancement sebagai Preprocessing untuk Object Detection pada Dataset ExDark**

| Scenario | Enhancement | Training Data | Test Data |
|----------|-------------|---------------|----------|
| S1_Raw | None (Baseline) | Raw | Raw |
| S2_HVI_CIDNet | HVI-CIDNet | Enhanced | Enhanced |
| S3_RetinexFormer | RetinexFormer | Enhanced | Enhanced |
| S4_LYT_Net | LYT-Net | Enhanced | Enhanced |

---
**How to use:**
1. Set `ACTIVE_SCENARIOS` di Cell 0.1 — pilih skenario yang mau dijalankan
2. **Ctrl+F9 (Run All)** — semua fase berjalan otomatis
3. Fase yang sudah selesai (output ada di Drive) → **auto-skip**
4. Fase 7 (Report) membaca **semua hasil yang tersimpan** di Drive, bukan hanya yang aktif

**Cache-first principle:** Setiap fase cek output di Drive. Jika sudah ada → skip. Jika belum → run. Jika mau ulang → set `force=True`.

## 0. Configuration & Environment Setup

In [ ]:
#@title 0.1 Mount Google Drive & Clone Repo
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
QUICK_TEST = True  # @param {type:"boolean"}
REPO_URL = "https://github.com/Otachiking/Object-Detection-ExDARK-with-LLIE.git"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KULIAH-S1INFORMATIKA/TA-IQBAL"  # @param {type:"string"}

# === SCENARIO SELECTOR ===
# Comment/uncomment to choose which scenarios to RUN this session.
# Already-completed scenarios auto-skip (cached on Drive).
# Final report (Fase 7) reads ALL saved results, not just active ones.
ACTIVE_SCENARIOS = [
    ("s1_raw", "S1_Raw"),
    # ("s2_hvi_cidnet", "S2_HVI_CIDNet"),
    # ("s3_retinexformer", "S3_RetinexFormer"),
    # ("s4_lyt_net", "S4_LYT_Net"),
]

# Full scenario registry (used by Fase 7 to discover completed results)
ALL_SCENARIOS = [
    ("s1_raw", "S1_Raw"),
    ("s2_hvi_cidnet", "S2_HVI_CIDNet"),
    ("s3_retinexformer", "S3_RetinexFormer"),
    ("s4_lyt_net", "S4_LYT_Net"),
]

SCENARIO_LABELS = {
    "S1_Raw": "S1: Baseline (Raw)",
    "S2_HVI_CIDNet": "S2: HVI-CIDNet",
    "S3_RetinexFormer": "S3: RetinexFormer",
    "S4_LYT_Net": "S4: LYT-Net",
}

# Quick lookup
ACTIVE_NAMES = {sn for _, sn in ACTIVE_SCENARIOS}

def is_active(scenario_name):
    """Check if a scenario is active this session."""
    return scenario_name in ACTIVE_NAMES

# Clone or pull repo
import os
import subprocess

REPO_DIR = "/content/TA-IQBAL-ObjectDetectionExDARKwithLLIE"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo exists, resetting to latest origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    if os.path.exists(REPO_DIR):
        import shutil
        shutil.rmtree(REPO_DIR)
    print(f"Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Quick test: {QUICK_TEST}")
print(f"Active scenarios: {[s[1] for s in ACTIVE_SCENARIOS]}")

assert os.path.isfile(os.path.join(REPO_DIR, "src", "data", "split_dataset.py")), \
    "src/data/split_dataset.py missing after clone!"
print("✓ Source files verified")

In [ ]:
#@title 0.2 Install Dependencies
!pip install -q ultralytics pyiqa thop fvcore scipy pandas pyyaml seaborn tqdm gdown huggingface_hub

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 0.3 Load Configuration
import os
import sys
sys.path.insert(0, REPO_DIR)

from src.config import load_config, save_config_snapshot, save_environment_info
from src.seed import set_global_seed

# Load S1 config (base) to get paths
cfg_s1 = load_config("s1_raw", quick_test=QUICK_TEST)
set_global_seed(cfg_s1["seed"])

# ---- Compatibility layer (old notebook keys vs new paths.yaml schema) ----
paths = cfg_s1.get("paths", {})
data_paths = paths.get("data", {})

# Main roots
OUTPUT_ROOT = paths.get("output_root") or paths.get("drive_root") or paths.get("project_root")
EXDARK_ROOT = paths.get("exdark_root") or data_paths.get("exdark_original")

if OUTPUT_ROOT is None:
    raise KeyError("Could not resolve OUTPUT_ROOT from config['paths']")
if EXDARK_ROOT is None:
    raise KeyError("Could not resolve EXDARK_ROOT from config['paths']['data']['exdark_original']")

# Backfill expected keys for downstream cells
cfg_s1.setdefault("paths", {})
cfg_s1["paths"]["output_root"] = OUTPUT_ROOT
cfg_s1["paths"]["exdark_root"] = EXDARK_ROOT

# Backfill old exdark_structure mapping if needed
if "exdark_structure" not in cfg_s1["paths"]:
    exdark_meta = cfg_s1.get("paths_meta", {}).get("exdark", {})
    cfg_s1["paths"]["exdark_structure"] = {
        "images": exdark_meta.get("images_dir", "Dataset/ExDark"),
        "groundtruth": exdark_meta.get("groundtruth_dir", "Groundtruth"),
        "classlist": exdark_meta.get("classlist_file", "Groundtruth/imageclasslist.txt"),
    }

print(f"Output root: {OUTPUT_ROOT}")
print(f"ExDark root: {EXDARK_ROOT}")
print(f"Seed: {cfg_s1['seed']}")

# Verify ExDark exists
assert os.path.exists(EXDARK_ROOT), f"ExDark not found at {EXDARK_ROOT}. Upload dataset to Drive first!"
print("\n✓ ExDark dataset found")

# Save environment info
os.makedirs(OUTPUT_ROOT, exist_ok=True)
save_environment_info(OUTPUT_ROOT)

---
## Fase 1: Data Preparation
Parse official split → Convert annotations → Build YOLO structure → Validate

In [ ]:
#@title Fase 1.1: Parse Official Split
import os
import sys

# Ensure repo is importable (REPO_DIR set by Cell 0.1)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data.split_dataset import parse_split_file

classlist_path = os.path.join(
    EXDARK_ROOT,
    cfg_s1["paths"]["exdark_structure"]["groundtruth"],
    "imageclasslist.txt"
)
split_output = os.path.join(OUTPUT_ROOT, "splits")

splits = parse_split_file(classlist_path, split_output)
print(f"Repo: {REPO_DIR}")
print(f"Train: {splits['train']} | Val: {splits['val']} | Test: {splits['test']}")
print(f"Total: {splits['train'] + splits['val'] + splits['test']}")


In [ ]:
#@title Fase 1.2: Convert Annotations to YOLO Format
from src.data.convert_exdark import convert_exdark_to_yolo

gt_dir = os.path.join(EXDARK_ROOT, cfg_s1["paths"]["exdark_structure"]["groundtruth"])
img_dir = os.path.join(EXDARK_ROOT, cfg_s1["paths"]["exdark_structure"]["images"])
yolo_labels_dir = os.path.join(OUTPUT_ROOT, "ExDark_yolo_labels")

stats = convert_exdark_to_yolo(
    exdark_images_dir=img_dir,
    exdark_gt_dir=gt_dir,
    output_labels_dir=yolo_labels_dir,
)
print(f"\nImages: {stats['total_images']} | Labels: {stats['total_labels']} | "
      f"Objects: {stats['total_objects']} | Failed: {stats['failed']}")


In [ ]:
#@title Fase 1.3: Build YOLO Dataset Structure
from src.data.build_yolo_dataset import build_yolo_dataset

yolo_dir = os.path.join(OUTPUT_ROOT, "ExDark_yolo")

build_stats = build_yolo_dataset(
    exdark_images_dir=img_dir,
    labels_dir=yolo_labels_dir,
    splits_dir=split_output,
    output_dir=yolo_dir,
    target_size=cfg_s1["yolo"]["imgsz"],
)

total_processed = sum(s["processed"] for s in build_stats["splits"].values())
total_skipped = sum(s["skipped"] for s in build_stats["splits"].values())
print(f"\nProcessed: {total_processed} | Skipped: {total_skipped} | Errors: {len(build_stats['errors'])}")
print(f"Dataset YAML: {os.path.join(yolo_dir, 'dataset.yaml')}")


In [ ]:
#@title Fase 1.4: Validate Dataset
from src.data.validate_dataset import validate_yolo_dataset

val_results = validate_yolo_dataset(yolo_dir)

if val_results["valid"]:
    print("\n✓ Dataset validation PASSED")
    summary = val_results.get("summary", {})
    print(f"  Total images:  {summary.get('total_images', 0)}")
    print(f"  Total labels:  {summary.get('total_labels', 0)}")
    print(f"  Total objects: {summary.get('total_objects', 0)}")
else:
    print("\n✗ Validation FAILED:")
    for split_name, sd in val_results.get("splits", {}).items():
        if sd.get("error"):
            print(f"  - {split_name}: {sd['error']}")
        if sd.get("invalid_bbox", 0) > 0:
            print(f"  - {split_name}: {sd['invalid_bbox']} invalid bboxes")
        if sd.get("missing_labels", 0) > 0:
            print(f"  - {split_name}: {sd['missing_labels']} images without labels")


---
## Fase 2: Image Enhancement
Apply LLIE methods to generate enhanced datasets. S1 (Raw) auto-skips.

In [ ]:
#@title Fase 2: Image Enhancement (loops active scenarios, auto-skips S1/completed)
import torch
from src.enhancement.run_enhancement import enhance_dataset, get_enhancer

enhancement_stats = {}

for sc, sn in ACTIVE_SCENARIOS:
    cfg = load_config(sc, quick_test=QUICK_TEST)
    set_global_seed(cfg["seed"])

    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    has_enhancer = enhancer_name and enhancer_name.lower() != "none"

    if not has_enhancer:
        print(f"\n{'='*50}")
        print(f"[SKIP] {sn}: No enhancement (baseline scenario)")
        print(f"{'='*50}")
        continue

    enhanced_dir = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}")
    cache_dir = os.path.join(OUTPUT_ROOT, "model_cache")

    print(f"\n{'='*50}")
    print(f"Enhancing: {sn} ({enhancer_name})")
    print(f"Output: {enhanced_dir}")
    print(f"{'='*50}")

    enhancer = get_enhancer(enhancer_name, cache_dir)
    enhancer.load_model()

    stats = enhance_dataset(
        enhancer=enhancer,
        source_dataset_dir=yolo_dir,
        output_dir=enhanced_dir,
        yolo_labels_dir=yolo_dir,
    )

    enhancement_stats[sn] = stats
    print(f"\nDone: {stats['total_processed']} processed, "
          f"{stats['total_skipped']} skipped, {stats['total_failed']} failed")

    # Free GPU memory
    del enhancer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*50}")
print(f"Fase 2 complete. Enhanced: {list(enhancement_stats.keys()) or ['(none — baseline only)']}")
print(f"{'='*50}")

---
## Fase 3: YOLOv11n Training
Train one model per scenario. Each uses the same hyperparameters (only data differs).

**Locked params:** epochs=100, batch=16, imgsz=640, patience=20, seed=42

In [ ]:
#@title Fase 3: YOLOv11n Training (loops active scenarios, auto-skips if best.pt exists)
from src.training.train_yolo import train_yolo, get_best_weights

training_results = {}

for sc, sn in ACTIVE_SCENARIOS:
    cfg = load_config(sc, quick_test=QUICK_TEST)
    set_global_seed(cfg["seed"])

    # Determine dataset YAML
    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    if enhancer_name and enhancer_name.lower() != "none":
        data_yaml = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}", "dataset.yaml")
    else:
        data_yaml = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "dataset.yaml")

    assert os.path.exists(data_yaml), f"Dataset YAML not found: {data_yaml}"

    runs_dir = os.path.join(OUTPUT_ROOT, "runs")

    print(f"\n{'='*50}")
    print(f"Training: {sn}")
    print(f"Data: {data_yaml}")
    print(f"Run dir: {runs_dir}/{sn}")
    print(f"{'='*50}")

    results = train_yolo(
        dataset_yaml=data_yaml,
        scenario_name=sn,
        output_dir=runs_dir,
        config=cfg,
    )

    training_results[sn] = results

    best = get_best_weights(os.path.join(runs_dir, sn))
    print(f"Best weights: {best}")

print(f"\n{'='*50}")
print(f"Fase 3 complete. Trained: {list(training_results.keys())}")
print(f"{'='*50}")

---
## Fase 4: Detection Evaluation
Evaluate each trained model on the test split.

**Metrics:** mAP@0.5, mAP@0.5:0.95, Precision, Recall (overall + per-class)

In [ ]:
#@title Fase 4: Detection Evaluation (loops active scenarios, auto-skips if metrics.json exists)
from src.evaluation.eval_yolo import evaluate_yolo
import pandas as pd

eval_results = {}

for sc, sn in ACTIVE_SCENARIOS:
    cfg = load_config(sc, quick_test=QUICK_TEST)
    set_global_seed(cfg["seed"])

    run_dir = os.path.join(OUTPUT_ROOT, "runs", sn)
    weights_path = get_best_weights(run_dir)
    assert os.path.exists(weights_path), f"Weights not found: {weights_path}"

    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    if enhancer_name and enhancer_name.lower() != "none":
        data_yaml = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}", "dataset.yaml")
    else:
        data_yaml = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "dataset.yaml")

    eval_dir = os.path.join(OUTPUT_ROOT, "evaluation", sn)

    print(f"\n--- {sn} ---")
    results = evaluate_yolo(
        weights_path=weights_path,
        dataset_yaml=data_yaml,
        output_dir=eval_dir,
        scenario_name=sn,
    )

    overall = results.get("overall", {})
    eval_results[sn] = overall
    print(f"  mAP@0.5:      {overall.get('mAP_50', 0):.4f}")
    print(f"  mAP@0.5:0.95: {overall.get('mAP_50_95', 0):.4f}")
    print(f"  Precision:     {overall.get('precision', 0):.4f}")
    print(f"  Recall:        {overall.get('recall', 0):.4f}")

# Summary table
summary_rows = []
for sn, res in eval_results.items():
    summary_rows.append({
        "Scenario": sn,
        "mAP@0.5": f"{res.get('mAP_50', 0):.4f}",
        "mAP@0.5:0.95": f"{res.get('mAP_50_95', 0):.4f}",
        "Precision": f"{res.get('precision', 0):.4f}",
        "Recall": f"{res.get('recall', 0):.4f}",
    })
display(pd.DataFrame(summary_rows))

---
## Fase 5: No-Reference Image Quality Assessment
Compute NIQE, BRISQUE, and LOE on enhanced test images.

**Note:** LOE requires raw-enhanced pairs, so it's only computed for S2-S4.

In [ ]:
#@title Fase 5: NR-IQA Metrics (active scenarios only)
from src.evaluation.nr_metrics import compute_nr_metrics

nr_results = {}
raw_test_dir = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "images", "test")

for sc, sn in ACTIVE_SCENARIOS:
    cfg = load_config(sc, quick_test=QUICK_TEST)
    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    has_enhancer = enhancer_name and enhancer_name.lower() != "none"

    if has_enhancer:
        test_dir = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}", "images", "test")
        raw_dir = raw_test_dir
    else:
        test_dir = raw_test_dir
        raw_dir = None

    nr_dir = os.path.join(OUTPUT_ROOT, "evaluation", sn, "nr_metrics")

    print(f"\n--- {sn} ---")
    nr_results[sn] = compute_nr_metrics(
        images_dir=test_dir,
        output_dir=nr_dir,
        scenario_name=sn,
        raw_images_dir=raw_dir,
    )

# Summary
nr_rows = []
for sn, res in nr_results.items():
    nr_rows.append({
        "Scenario": sn,
        "NIQE ↓": res.get("niqe_mean", None),
        "BRISQUE ↓": res.get("brisque_mean", None),
        "LOE ↓": res.get("loe_mean", None),
    })
pd.DataFrame(nr_rows)


---
## Fase 6: Latency & FLOPs Measurement
Measure end-to-end inference speed and computational cost.

In [ ]:
#@title Fase 6: Latency & FLOPs (loops active scenarios, auto-skips if cached)
from src.evaluation.latency import measure_latency
from src.evaluation.flops import compute_all_flops

latency_results = {}
flops_results = {}

for sc, sn in ACTIVE_SCENARIOS:
    cfg = load_config(sc, quick_test=QUICK_TEST)
    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    has_enhancer = enhancer_name and enhancer_name.lower() != "none"

    run_dir = os.path.join(OUTPUT_ROOT, "runs", sn)
    weights_path = get_best_weights(run_dir)

    # --- Latency ---
    lat_dir = os.path.join(OUTPUT_ROOT, "evaluation", sn, "latency")

    enhancer_obj = None
    if has_enhancer:
        cache_dir = os.path.join(OUTPUT_ROOT, "model_cache")
        enhancer_obj = get_enhancer(enhancer_name, cache_dir)
        enhancer_obj.load_model()

    print(f"\n--- {sn}: Latency ---")
    raw_lat = measure_latency(
        yolo_weights=weights_path,
        output_dir=lat_dir,
        scenario_name=sn,
        test_images_dir=raw_test_dir,
        enhancer=enhancer_obj,
        num_images=cfg.get("latency", {}).get("iterations", 200),
        warmup=cfg.get("latency", {}).get("warmup", 50),
    )

    latency_results[sn] = {
        "T_enhance_mean": raw_lat.get("T_enhance_ms_mean", 0),
        "T_detect_mean": raw_lat.get("T_detect_ms_mean", 0),
        "T_total_mean": raw_lat.get("T_total_ms_mean", 0),
    }

    # --- FLOPs ---
    flops_dir = os.path.join(OUTPUT_ROOT, "evaluation", sn, "flops")

    enhancer_model = None
    if has_enhancer and enhancer_obj is not None:
        enhancer_model = enhancer_obj.model

    print(f"\n--- {sn}: FLOPs ---")
    raw_fl = compute_all_flops(
        yolo_weights=weights_path,
        output_dir=flops_dir,
        scenario_name=sn,
        enhancer_model=enhancer_model,
        enhancer_name=enhancer_name if has_enhancer else None,
    )

    flops_results[sn] = {
        "enhancer_gflops": raw_fl.get("enhancer", {}).get("gflops", 0) or 0,
        "yolo_gflops": raw_fl.get("yolo", {}).get("gflops", 0) or 0,
        "total_gflops": raw_fl.get("total_gflops", 0) or 0,
    }

    # Free GPU memory
    if enhancer_obj:
        del enhancer_obj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Summary
print(f"\n{'='*50}")
print("Latency Summary:")
lat_rows = []
for sn, res in latency_results.items():
    lat_rows.append({
        "Scenario": sn,
        "T_enhance (ms)": f"{res['T_enhance_mean']:.2f}",
        "T_detect (ms)": f"{res['T_detect_mean']:.2f}",
        "T_total (ms)": f"{res['T_total_mean']:.2f}",
    })
display(pd.DataFrame(lat_rows))

print("\nFLOPs Summary:")
flops_rows = []
for sn, res in flops_results.items():
    flops_rows.append({
        "Scenario": sn,
        "Enhancer GFLOPs": f"{res['enhancer_gflops']:.2f}",
        "YOLO GFLOPs": f"{res['yolo_gflops']:.2f}",
        "Total GFLOPs": f"{res['total_gflops']:.2f}",
    })
display(pd.DataFrame(flops_rows))

---
## Fase 7: Aggregation, Correlation & Visualization

**Reads ALL saved results from Drive** — not limited to `ACTIVE_SCENARIOS`.
If you've already completed S1 yesterday and S2 today, Fase 7 will include both.

In [ ]:
#@title Fase 7a: Load ALL Saved Results from Drive
import json

def load_json_safe(path):
    """Load JSON file, return None if not found."""
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return None

# Discover which scenarios have completed results on Drive
completed_scenarios = []
all_eval = {}
all_nr = {}
all_latency = {}
all_flops = {}

print("Scanning Drive for completed scenario results...\n")

for sc, sn in ALL_SCENARIOS:
    eval_path = os.path.join(OUTPUT_ROOT, "evaluation", sn, "metrics.json")
    nr_path = os.path.join(OUTPUT_ROOT, "evaluation", sn, "nr_metrics", "summary.json")
    lat_path = os.path.join(OUTPUT_ROOT, "evaluation", sn, "latency", "latency.json")
    flops_path = os.path.join(OUTPUT_ROOT, "evaluation", sn, "flops", "flops.json")

    eval_data = load_json_safe(eval_path)
    nr_data = load_json_safe(nr_path)
    lat_data = load_json_safe(lat_path)
    flops_data = load_json_safe(flops_path)

    has_eval = eval_data is not None
    has_weights = os.path.exists(os.path.join(OUTPUT_ROOT, "runs", sn, "weights", "best.pt"))

    status_parts = []
    if has_weights:
        status_parts.append("✓ weights")
    if has_eval:
        status_parts.append("✓ eval")
        all_eval[sn] = eval_data.get("overall", eval_data)
    if nr_data:
        status_parts.append("✓ NR-IQA")
        all_nr[sn] = nr_data
    if lat_data:
        status_parts.append("✓ latency")
        all_latency[sn] = {
            "T_enhance_mean": lat_data.get("T_enhance_ms_mean", 0),
            "T_detect_mean": lat_data.get("T_detect_ms_mean", 0),
            "T_total_mean": lat_data.get("T_total_ms_mean", 0),
        }
    if flops_data:
        status_parts.append("✓ FLOPs")
        all_flops[sn] = {
            "enhancer_gflops": flops_data.get("enhancer", {}).get("gflops", 0) or 0,
            "yolo_gflops": flops_data.get("yolo", {}).get("gflops", 0) or 0,
            "total_gflops": flops_data.get("total_gflops", 0) or 0,
        }

    if has_eval:
        completed_scenarios.append((sc, sn))

    status = " | ".join(status_parts) if status_parts else "✗ not found"
    active_tag = " ← ACTIVE" if sn in ACTIVE_NAMES else ""
    print(f"  {sn}: {status}{active_tag}")

print(f"\n{'='*50}")
print(f"Completed scenarios (have eval): {[sn for _, sn in completed_scenarios]}")
print(f"Active this session: {[sn for _, sn in ACTIVE_SCENARIOS]}")
print(f"{'='*50}")

In [ ]:
#@title Fase 7b: Spearman Correlation (uses ALL completed scenarios)
from src.evaluation.correlation import compute_spearman_correlation

completed_names = [sn for _, sn in completed_scenarios]
corr_results = {}

if len(completed_names) >= 3:
    corr_dir = os.path.join(OUTPUT_ROOT, "summary", "correlation")
    corr_results = compute_spearman_correlation(
        detection_results=all_eval,
        nr_results=all_nr,
        output_dir=corr_dir,
    )

    if "correlations" in corr_results:
        corr_rows = []
        for entry in corr_results["correlations"]:
            corr_rows.append({
                "NR Metric": entry["nr_metric"],
                "Det Metric": entry["det_metric"],
                "Spearman ρ": f"{entry['spearman_rho']:.4f}" if entry['spearman_rho'] else "N/A",
                "p-value": f"{entry['p_value']:.4f}" if entry['p_value'] else "N/A",
                "Interpretation": entry["interpretation"],
            })
        display(pd.DataFrame(corr_rows))
else:
    print(f"⚠ Correlation requires ≥3 completed scenarios. Found: {len(completed_names)}")
    print(f"  Completed: {completed_names}")
    print(f"  → Run more scenarios, then re-run Fase 7.")

In [ ]:
#@title Fase 7c: Generate All Figures (uses ALL completed scenarios)
from src.utils.visualization import (
    plot_detection_comparison,
    plot_nr_metrics,
    plot_latency_breakdown,
    plot_correlation_scatter,
    export_detection_latex,
)

figures_dir = os.path.join(OUTPUT_ROOT, "summary", "figures")
os.makedirs(figures_dir, exist_ok=True)

# Detection comparison bar chart
if all_eval:
    plot_detection_comparison(
        all_eval,
        os.path.join(figures_dir, "detection_comparison.png"),
    )

# NR metrics comparison
if all_nr:
    plot_nr_metrics(
        all_nr,
        os.path.join(figures_dir, "nr_metrics.png"),
    )

# Latency breakdown
if all_latency:
    plot_latency_breakdown(
        all_latency,
        os.path.join(figures_dir, "latency_breakdown.png"),
    )

# Correlation scatter plots (only if ≥3 scenarios)
if corr_results and "correlations" in corr_results:
    merged = {}
    for sn in completed_names:
        if sn in all_eval and sn in all_nr:
            merged[sn] = {**all_eval[sn], **all_nr[sn]}

    for entry in corr_results["correlations"]:
        plot_correlation_scatter(
            data=merged,
            nr_metric=entry["nr_metric"],
            det_metric=entry["det_metric"],
            output_path=os.path.join(figures_dir, f"corr_{entry['nr_metric']}_vs_{entry['det_metric']}.png"),
            rho=entry.get("spearman_rho"),
            p_value=entry.get("p_value"),
        )

# LaTeX table
if all_eval:
    export_detection_latex(
        all_eval,
        os.path.join(OUTPUT_ROOT, "summary", "table_detection.tex"),
    )

print(f"\nAll figures saved in: {figures_dir}")

In [ ]:
#@title Fase 7e: Visual Detection Comparison (4 same test images × all completed scenarios)
import cv2
import numpy as np
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

CLASS_NAMES_VIS = {
    0: "Bicycle", 1: "Boat", 2: "Bottle", 3: "Bus",
    4: "Car", 5: "Cat", 6: "Chair", 7: "Cup",
    8: "Dog", 9: "Motorbike", 10: "People", 11: "Table",
}

CLS_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8",
    "#f58231", "#911eb4", "#42d4f4", "#f032e6",
    "#bfef45", "#fabed4", "#469990", "#dcbeff",
]

NUM_IMAGES = 4
VIS_SEED = 42
CONF_THRESH = 0.25

random.seed(VIS_SEED)

# Pick 4 random test images (deterministic seed, same across scenarios)
raw_test_dir_vis = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "images", "test")
test_images_all = sorted([f for f in os.listdir(raw_test_dir_vis)
                          if f.lower().endswith((".jpg", ".jpeg", ".png"))])
selected_vis = random.sample(test_images_all, min(NUM_IMAGES, len(test_images_all)))

print(f"Selected test images: {selected_vis}")
print(f"Scenarios to compare: {completed_names}")

# Load YOLO models for completed scenarios
vis_models = {}
all_sc_dict = dict(ALL_SCENARIOS)
for sc, sn in completed_scenarios:
    w = os.path.join(OUTPUT_ROOT, "runs", sn, "weights", "best.pt")
    if os.path.exists(w):
        vis_models[sn] = YOLO(w)
        print(f"  Loaded model: {sn}")

n_sc = len(vis_models)
if n_sc == 0:
    print("No trained models found. Skipping visualization.")
else:
    scenario_order = [sn for sn in [s[1] for s in ALL_SCENARIOS] if sn in vis_models]

    fig, axes = plt.subplots(NUM_IMAGES, n_sc, figsize=(5 * n_sc, 5 * NUM_IMAGES))
    if NUM_IMAGES == 1 and n_sc == 1:
        axes = np.array([[axes]])
    elif NUM_IMAGES == 1:
        axes = axes.reshape(1, -1)
    elif n_sc == 1:
        axes = axes.reshape(-1, 1)

    for row_idx, img_name in enumerate(selected_vis):
        for col_idx, sn in enumerate(scenario_order):
            ax = axes[row_idx, col_idx]

            # Determine image path (raw for S1, enhanced for S2-S4)
            cfg_vis = load_config(all_sc_dict[sn], quick_test=QUICK_TEST)
            enh_name = cfg_vis.get("enhancer", {}).get("name", None)
            has_enh = enh_name and enh_name.lower() != "none"

            if has_enh:
                vis_img_dir = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enh_name}", "images", "test")
            else:
                vis_img_dir = raw_test_dir_vis

            img_path = os.path.join(vis_img_dir, img_name)
            if not os.path.exists(img_path):
                ax.text(0.5, 0.5, f"Not found:\n{img_name}", ha='center', va='center',
                        transform=ax.transAxes, fontsize=9)
                ax.axis('off')
                continue

            img_bgr = cv2.imread(img_path)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            # Run detection
            det_results = vis_models[sn](img_bgr, imgsz=640, conf=CONF_THRESH, verbose=False)

            ax.imshow(img_rgb)

            # Draw bounding boxes with confidence scores
            if det_results and len(det_results[0].boxes) > 0:
                for box in det_results[0].boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    cls_id = int(box.cls[0].cpu().numpy())
                    conf = float(box.conf[0].cpu().numpy())
                    color = CLS_COLORS[cls_id % len(CLS_COLORS)]
                    label = f"{CLASS_NAMES_VIS.get(cls_id, str(cls_id))} {conf:.2f}"

                    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                             linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, y1 - 3, label, fontsize=7, color='white',
                            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

            # Title on top row, image name on left column
            if row_idx == 0:
                ax.set_title(SCENARIO_LABELS.get(sn, sn), fontsize=12, fontweight='bold')
            if col_idx == 0:
                ax.set_ylabel(img_name.replace('.jpg', ''), fontsize=9, rotation=90, labelpad=10)
            ax.axis('off')

    plt.tight_layout(pad=1.5)
    vis_out_path = os.path.join(figures_dir, "visual_detection_comparison.png")
    plt.savefig(vis_out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSaved: {vis_out_path}")

    del vis_models
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

---
## Visual Detection Comparison & Final Summary

In [ ]:
#@title Fase 7d: Save & Display Final Comparison Table (ALL completed scenarios)

# Compile master summary
summary_dir = os.path.join(OUTPUT_ROOT, "summary")
os.makedirs(summary_dir, exist_ok=True)

master_summary = {
    "quick_test": QUICK_TEST,
    "completed_scenarios": completed_names,
    "active_this_session": [sn for _, sn in ACTIVE_SCENARIOS],
    "scenarios": {},
}

final_rows = []
for sn in completed_names:
    det = all_eval.get(sn, {})
    nr = all_nr.get(sn, {})
    lat = all_latency.get(sn, {})
    fl = all_flops.get(sn, {})

    master_summary["scenarios"][sn] = {
        "detection": det,
        "nr_metrics": nr,
        "latency": lat,
        "flops": fl,
    }

    final_rows.append({
        "Scenario": SCENARIO_LABELS.get(sn, sn),
        "mAP@0.5": f"{det.get('mAP_50', 0):.4f}",
        "mAP@0.5:0.95": f"{det.get('mAP_50_95', 0):.4f}",
        "Precision": f"{det.get('precision', 0):.4f}",
        "Recall": f"{det.get('recall', 0):.4f}",
        "NIQE ↓": f"{nr.get('niqe_mean', '-')}",
        "BRISQUE ↓": f"{nr.get('brisque_mean', '-')}",
        "T_total (ms)": f"{lat.get('T_total_mean', '-')}",
        "GFLOPs": f"{fl.get('total_gflops', '-')}",
    })

# Save
summary_path = os.path.join(summary_dir, "master_summary.json")
with open(summary_path, "w") as f:
    json.dump(master_summary, f, indent=2, default=str)
print(f"Master summary saved: {summary_path}")

# Display
df_final = pd.DataFrame(final_rows)
print(f"\n{'='*80}")
print(f"FINAL COMPARISON TABLE ({len(completed_names)} scenarios)")
print(f"{'='*80}")
display(df_final)

if len(completed_names) < 4:
    missing = [sn for _, sn in ALL_SCENARIOS if sn not in completed_names]
    print(f"\n⚠ Missing scenarios: {missing}")
    print(f"  → Add them to ACTIVE_SCENARIOS, Ctrl+F9, then re-run Fase 7.")